# Caso 07 — Ampara, seguros: ¿qué canal paga lo que cuesta?
## Día 3 (Mañana y Tarde) — Del Ranking de Superficie a la Primera Grieta: el Loss Ratio

Con las 7 tablas limpias en `data/clean/`, pasamos a **SQL (DuckDB)**, que es el lenguaje del almacén y de los cuadros de mando de Ampara.

### Estructura del Día 3:
- **Mañana — El ranking de superficie (lo que mira Distribución):**
  1. Pólizas nuevas, prima suscrita total, cuotas sobre el total (`SUM(...) OVER ()`) y prima media por canal (`WHERE tipo = 'nueva' AND prima_anual > 0`).
  2. **CAC por canal** usando el **patrón de las dos piernas (CTEs separadas)** para no multiplicar filas al cruzar `campanas` con `polizas`.
  3. **Variables de contexto:** reparto por ramo dentro de cada canal (la semilla de la corazonada de Lucía: `comparadores` vende un 60% de `auto`), provincia y descuento (uniformes entre canales), edad media del cliente (comparadores capta más joven, 34 años vs 44 en banca) y cruce con `mediadores` mediante `LEFT JOIN`.
- **Tarde — La primera grieta: el Loss Ratio:**
  4. **Loss Ratio por canal** agregando siniestros y prima en dos CTEs separadas y conservando los siniestros abiertos (`comparadores` sube al **82,0%** frente al **48,0%** de `referido`).
  5. **Demostración de las dos trampas:** qué pasaría si tirásemos los siniestros abiertos (`fecha_cierre IS NOT NULL` baja artificialmente el loss ratio de comparadores al 62,5%).
  6. **Descomposición del Loss Ratio:** frecuencia (siniestros por póliza), severidad (importe medio por siniestro) y porcentaje de pólizas sin siniestro (`LEFT JOIN`).
  7. **Descarte del fraude como explicación:** tasa de fraude por canal (~5	ext{--}6%) y recálculo del loss ratio sin fraude (la brecha de $>30$ puntos se mantiene intacta).


In [1]:
import duckdb
import pandas as pd
from pathlib import Path

con = duckdb.connect()
for tabla in ["canales", "productos", "mediadores", "campanas", "clientes", "polizas", "siniestros"]:
    con.execute(f"CREATE OR REPLACE VIEW {tabla} AS SELECT * FROM read_csv_auto('data/clean/{tabla}.csv')")

print("Vistas SQL registradas en DuckDB correctamente.")


Vistas SQL registradas en DuckDB correctamente.


### Parte I (Mañana): Las Cuatro Caras de la Vanidad (Pólizas, Prima Suscrita, Prima Media y CAC)
Calculamos en SQL usando CTEs y funciones de ventana (`OVER ()`) el cuadro de mando que mira Álex Cordero cada mañana.


In [2]:
query_superficie = """
WITH polizas_canal AS (
    SELECT
        canal,
        COUNT(*) AS polizas_nuevas,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS cuota_polizas_pct,
        ROUND(SUM(prima_anual), 2) AS prima_suscrita_eur,
        ROUND(100.0 * SUM(prima_anual) / SUM(SUM(prima_anual)) OVER (), 2) AS cuota_prima_pct,
        ROUND(AVG(prima_anual), 2) AS prima_media_eur
    FROM polizas
    WHERE tipo = 'nueva' AND prima_anual > 0
    GROUP BY canal
),
coste_canal AS (
    SELECT
        canal,
        ROUND(SUM(coste), 2) AS coste_campanas_eur
    FROM campanas
    GROUP BY canal
)
SELECT
    p.canal,
    p.polizas_nuevas,
    p.cuota_polizas_pct,
    p.prima_suscrita_eur,
    p.cuota_prima_pct,
    p.prima_media_eur,
    c.coste_campanas_eur,
    ROUND(c.coste_campanas_eur / p.polizas_nuevas, 2) AS cac_eur
FROM polizas_canal p
LEFT JOIN coste_canal c ON p.canal = c.canal
ORDER BY p.polizas_nuevas DESC
"""
df_superficie = con.execute(query_superficie).df()
display(df_superficie)


,canal,polizas_nuevas,cuota_polizas_pct,prima_suscrita_eur,cuota_prima_pct,prima_media_eur,coste_campanas_eur,cac_eur
0,comparadores,24000,40.0,11520000.07,40.25,480.0,360000.0,15.0
1,directo,13200,22.0,5543999.73,19.37,420.0,396000.0,30.0
2,mediadores,10800,18.0,6587999.95,23.02,610.0,972000.0,90.0
3,banca,7200,12.0,2807999.68,9.81,390.0,324000.0,45.0
4,referido,4800,8.0,2159999.95,7.55,450.0,96000.0,20.0


### Contexto de Superficie: Mix de Ramo (la corazonada de Lucía), Edad, Provincia, Descuento y Mediadores
1. **Mix de ramo por canal:** verificamos cuánto pesa `auto` dentro de cada canal.
2. **Edad media y descuento medio por canal:** comprobamos que `comparadores` atrae al cliente más joven (34 años) mientras el descuento es uniforme (~6,5%).
3. **Provincia y Mediadores (`LEFT JOIN`):** comprobamos que la distribución geográfica es uniforme y que al cruzar con `mediadores` usando `LEFT JOIN` conservamos el 68% de ventas digitales (`comparadores` y `directo`).


In [3]:
# 1. Mix de ramo dentro de cada canal
query_ramo_canal = """
SELECT
    p.canal,
    pr.ramo,
    COUNT(*) AS polizas,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY p.canal), 2) AS pct_dentro_canal
FROM polizas p
JOIN productos pr ON p.producto_id = pr.producto_id
WHERE p.tipo = 'nueva' AND p.prima_anual > 0
GROUP BY p.canal, pr.ramo
ORDER BY p.canal, polizas DESC
"""
df_ramo_canal = con.execute(query_ramo_canal).df()
piv_ramo = df_ramo_canal.pivot(index="canal", columns="ramo", values="pct_dentro_canal")[["auto", "hogar", "salud", "vida", "mascotas"]]
print("=== % DE PÓLIZAS NUEVAS POR RAMO DENTRO DE CADA CANAL ===")
display(piv_ramo)

# 2. Edad media del cliente y descuento medio por canal
query_contexto_cliente = """
SELECT
    p.canal,
    COUNT(*) AS polizas_nuevas,
    ROUND(AVG(c.edad), 1) AS edad_media_cliente,
    ROUND(AVG(p.descuento_pct), 2) AS descuento_medio_pct,
    COUNT(m.mediador_id) AS polizas_con_mediador,
    ROUND(100.0 * COUNT(m.mediador_id) / COUNT(*), 1) AS pct_con_mediador
FROM polizas p
JOIN clientes c ON p.cliente_id = c.cliente_id
LEFT JOIN mediadores m ON p.mediador_id = m.mediador_id
WHERE p.tipo = 'nueva' AND p.prima_anual > 0
GROUP BY p.canal
ORDER BY polizas_nuevas DESC
"""
print("\n=== CONTEXTO POR CANAL: EDAD MEDIA, DESCUENTO Y PRESENCIA DE MEDIADOR (LEFT JOIN) ===")
display(con.execute(query_contexto_cliente).df())

# 3. Distribución por provincia dentro de cada canal (Top 5 provincias)
query_provincia = """
SELECT
    p.canal,
    c.provincia,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY p.canal), 1) AS pct_canal
FROM polizas p
JOIN clientes c ON p.cliente_id = c.cliente_id
WHERE p.tipo = 'nueva' AND p.prima_anual > 0
GROUP BY p.canal, c.provincia
"""
df_prov = con.execute(query_provincia).df()
print("\n=== DISTRIBUCIÓN POR PROVINCIA DENTRO DE CADA CANAL (%) - COMPROBACIÓN DE UNIFORMIDAD ===")
display(df_prov.pivot(index="canal", columns="provincia", values="pct_canal"))


=== % DE PÓLIZAS NUEVAS POR RAMO DENTRO DE CADA CANAL ===


ramo,auto,hogar,salud,vida,mascotas
canal,,,,,
banca,30.0,28.0,22.0,12.0,8.0
comparadores,60.0,15.0,12.0,5.0,8.0
directo,45.0,20.0,18.0,9.0,8.0
mediadores,25.0,30.0,20.0,15.0,10.0
referido,35.0,25.0,20.0,10.0,10.0



=== CONTEXTO POR CANAL: EDAD MEDIA, DESCUENTO Y PRESENCIA DE MEDIADOR (LEFT JOIN) ===


,canal,polizas_nuevas,edad_media_cliente,descuento_medio_pct,polizas_con_mediador,pct_con_mediador
0,comparadores,24000,33.9,7.75,0,0.0
1,directo,13200,37.7,7.65,0,0.0
2,mediadores,10800,43.3,7.68,10232,94.7
3,banca,7200,44.4,7.70,4653,64.6
4,referido,4800,40.4,7.70,1910,39.8



=== DISTRIBUCIÓN POR PROVINCIA DENTRO DE CADA CANAL (%) - COMPROBACIÓN DE UNIFORMIDAD ===


provincia,Barcelona,Bilbao,Bio,Madrid,Mur,Murcia,Málaga,Sevilla,Valencia,Zaragoza,desconocido
canal,,,,,,,,,,,
banca,17.3,6.6,3.5,21.8,1.9,3.7,9.7,11.5,13.5,7.8,2.9
comparadores,17.6,6.8,3.2,21.5,1.9,3.7,9.7,11.2,13.7,7.8,3.0
directo,17.6,6.3,3.0,21.5,2.0,3.9,9.9,11.7,13.5,7.6,2.9
mediadores,17.4,6.5,3.3,20.9,2.0,4.4,9.2,12.1,13.9,7.5,3.0
referido,16.8,6.7,2.9,22.2,1.8,4.1,9.4,13.1,12.4,7.2,3.4


---
### Parte II (Tarde): La Primera Grieta — El Loss Ratio por Canal
Cruzar `polizas` con `siniestros` exige evitar las **dos trampas** de las que avisó Hugo:
1. **No multiplicar la prima:** como una póliza puede tener varios siniestros, sumamos la prima en una CTE (`prima_canal`) y los siniestros en otra (`siniestros_canal`), y dividimos al final.
2. **Conservar los siniestros abiertos (`fecha_cierre IS NULL`):** su importe reservado (*incurrido*) ya es un coste real y concentra los siniestros más caros y lentos. Comparamos el loss ratio real (con abiertos) frente al sesgado (solo cerrados).


In [4]:
query_loss_ratio = """
WITH polizas_nuevas AS (
    SELECT poliza_id, canal, prima_anual
    FROM polizas
    WHERE tipo = 'nueva' AND prima_anual > 0
),
prima_canal AS (
    SELECT
        canal,
        COUNT(*) AS polizas_nuevas,
        SUM(prima_anual) AS prima_total_eur
    FROM polizas_nuevas
    GROUP BY canal
),
siniestros_canal AS (
    SELECT
        p.canal,
        COUNT(s.siniestro_id) AS total_siniestros,
        SUM(s.importe) AS siniestros_totales_eur,
        SUM(CASE WHEN s.fecha_cierre IS NOT NULL THEN s.importe ELSE 0 END) AS siniestros_solo_cerrados_eur,
        SUM(CASE WHEN s.fecha_cierre IS NULL THEN s.importe ELSE 0 END) AS siniestros_abiertos_eur
    FROM polizas_nuevas p
    LEFT JOIN siniestros s ON p.poliza_id = s.poliza_id
    GROUP BY p.canal
)
SELECT
    pr.canal,
    pr.polizas_nuevas,
    ROUND(pr.prima_total_eur, 2) AS prima_total_eur,
    sc.total_siniestros,
    ROUND(sc.siniestros_totales_eur, 2) AS siniestros_totales_eur,
    ROUND(100.0 * sc.siniestros_totales_eur / pr.prima_total_eur, 2) AS loss_ratio_real_pct,
    ROUND(100.0 * sc.siniestros_solo_cerrados_eur / pr.prima_total_eur, 2) AS loss_ratio_sin_abiertos_pct,
    ROUND(100.0 * (sc.siniestros_totales_eur - sc.siniestros_solo_cerrados_eur) / pr.prima_total_eur, 2) AS peso_abiertos_pp
FROM prima_canal pr
LEFT JOIN siniestros_canal sc ON pr.canal = sc.canal
ORDER BY loss_ratio_real_pct DESC
"""
df_loss = con.execute(query_loss_ratio).df()
display(df_loss)


,canal,polizas_nuevas,prima_total_eur,total_siniestros,siniestros_totales_eur,loss_ratio_real_pct,loss_ratio_sin_abiertos_pct,peso_abiertos_pp
0,comparadores,24000,11520000.07,53002,9446400.50,82.0,62.47,19.53
1,directo,13200,5543999.73,27048,3769919.67,68.0,56.50,11.50
2,banca,7200,2807999.68,13545,1628639.81,58.0,49.35,8.65
3,mediadores,10800,6587999.95,19364,3557519.86,54.0,46.47,7.53
4,referido,4800,2159999.95,9520,1036800.16,48.0,41.69,6.31


### Descomposición del Loss Ratio: Frecuencia, Severidad y Pólizas sin Siniestro (`LEFT JOIN`)
Analizamos por qué `comparadores` dispara su *loss ratio* al **82%**: ¿siniestra más a menudo (frecuencia), más caro cada vez (severidad) o tiene menos pólizas sin ningún siniestro?


In [5]:
query_frecuencia_severidad = """
WITH polizas_nuevas AS (
    SELECT poliza_id, canal, prima_anual
    FROM polizas
    WHERE tipo = 'nueva' AND prima_anual > 0
),
siniestros_por_poliza AS (
    SELECT
        p.poliza_id,
        p.canal,
        p.prima_anual,
        COUNT(s.siniestro_id) AS n_siniestros,
        COALESCE(SUM(s.importe), 0) AS importe_poliza
    FROM polizas_nuevas p
    LEFT JOIN siniestros s ON p.poliza_id = s.poliza_id
    GROUP BY p.poliza_id, p.canal, p.prima_anual
)
SELECT
    canal,
    COUNT(*) AS polizas_nuevas,
    SUM(CASE WHEN n_siniestros = 0 THEN 1 ELSE 0 END) AS polizas_sin_siniestro,
    ROUND(100.0 * SUM(CASE WHEN n_siniestros = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_polizas_sin_siniestro,
    SUM(n_siniestros) AS total_siniestros,
    ROUND(1.0 * SUM(n_siniestros) / COUNT(*), 2) AS frecuencia_siniestros_poliza,
    ROUND(SUM(importe_poliza) / NULLIF(SUM(n_siniestros), 0), 2) AS severidad_importe_medio_eur,
    ROUND(AVG(prima_anual), 2) AS prima_media_eur,
    ROUND(100.0 * SUM(importe_poliza) / SUM(prima_anual), 2) AS loss_ratio_pct
FROM siniestros_por_poliza
GROUP BY canal
ORDER BY loss_ratio_pct DESC
"""
df_frec_sev = con.execute(query_frecuencia_severidad).df()
display(df_frec_sev)


,canal,polizas_nuevas,polizas_sin_siniestro,pct_polizas_sin_siniestro,total_siniestros,frecuencia_siniestros_poliza,severidad_importe_medio_eur,prima_media_eur,loss_ratio_pct
0,comparadores,24000,5649.0,23.54,53002.0,2.21,178.23,480.0,82.0
1,directo,13200,3673.0,27.83,27048.0,2.05,139.38,420.0,68.0
2,banca,7200,2313.0,32.13,13545.0,1.88,120.24,390.0,58.0
3,mediadores,10800,3687.0,34.14,19364.0,1.79,183.72,610.0,54.0
4,referido,4800,1418.0,29.54,9520.0,1.98,108.91,450.0,48.0


### Descartar el Fraude: El Adorno que No Explica Nada
Comprobamos si la siniestralidad de los comparadores se explica por el fraude:
1. Medimos la **tasa de fraude** sobre los siniestros de cada canal.
2. Recalculamos el **loss ratio excluyendo los siniestros con `fraude = TRUE`**.


In [6]:
query_fraude = """
WITH polizas_nuevas AS (
    SELECT poliza_id, canal, prima_anual
    FROM polizas
    WHERE tipo = 'nueva' AND prima_anual > 0
),
prima_canal AS (
    SELECT canal, SUM(prima_anual) AS prima_total
    FROM polizas_nuevas
    GROUP BY canal
),
siniestros_canal AS (
    SELECT
        p.canal,
        COUNT(s.siniestro_id) AS total_siniestros,
        SUM(CASE WHEN s.fraude THEN 1 ELSE 0 END) AS siniestros_fraude,
        ROUND(100.0 * SUM(CASE WHEN s.fraude THEN 1 ELSE 0 END) / COUNT(s.siniestro_id), 2) AS tasa_fraude_pct,
        SUM(s.importe) AS importe_total,
        SUM(CASE WHEN NOT s.fraude THEN s.importe ELSE 0 END) AS importe_sin_fraude
    FROM polizas_nuevas p
    JOIN siniestros s ON p.poliza_id = s.poliza_id
    GROUP BY p.canal
)
SELECT
    pr.canal,
    sc.total_siniestros,
    sc.siniestros_fraude,
    sc.tasa_fraude_pct,
    ROUND(100.0 * sc.importe_total / pr.prima_total, 2) AS loss_ratio_con_fraude_pct,
    ROUND(100.0 * sc.importe_sin_fraude / pr.prima_total, 2) AS loss_ratio_sin_fraude_pct,
    ROUND(100.0 * (sc.importe_total - sc.importe_sin_fraude) / pr.prima_total, 2) AS impacto_fraude_pp
FROM prima_canal pr
JOIN siniestros_canal sc ON pr.canal = sc.canal
ORDER BY loss_ratio_con_fraude_pct DESC
"""
df_fraude = con.execute(query_fraude).df()
display(df_fraude)


,canal,total_siniestros,siniestros_fraude,tasa_fraude_pct,loss_ratio_con_fraude_pct,loss_ratio_sin_fraude_pct,impacto_fraude_pp
0,comparadores,53002,3166.0,5.97,82.0,76.94,5.06
1,directo,27048,1412.0,5.22,68.0,64.35,3.65
2,banca,13545,746.0,5.51,58.0,54.81,3.19
3,mediadores,19364,974.0,5.03,54.0,51.37,2.63
4,referido,9520,471.0,4.95,48.0,45.69,2.31
